# EDA - NASA CMAPSS FD001
This notebook documents data loading, RUL labeling, variance screening, and core exploratory plots.

In [ ]:
from pathlib import Path
import sys

ROOT = Path('..').resolve()
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import matplotlib.pyplot as plt
import seaborn as sns
from predictive_maintenance import prepare_train_test_with_rul, detect_near_zero_variance_sensors, SENSOR_COLS, RUL_COL

sns.set_theme(style='whitegrid')

In [ ]:
train_df, test_df = prepare_train_test_with_rul(ROOT / 'data', subset='FD001')
train_df.shape, test_df.shape

In [ ]:
dropped = detect_near_zero_variance_sensors(train_df, threshold=1e-4)
active = [s for s in SENSOR_COLS if s not in dropped]
dropped

In [ ]:
corr = train_df[[*active, RUL_COL]].corr(numeric_only=True)[RUL_COL].drop(RUL_COL).abs().sort_values(ascending=False)
top = corr.head(8).index.tolist()
top

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9), constrained_layout=True)
sampled = train_df['engine_id'].drop_duplicates().sample(10, random_state=42)
for idx, sensor in enumerate(top[:6]):
    ax = axes.flatten()[idx]
    for _, g in train_df[train_df['engine_id'].isin(sampled)].groupby('engine_id'):
        ax.plot(g['cycle'], g[sensor], alpha=0.35)
    ax.set_title(sensor)
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(train_df[RUL_COL], bins=40, kde=True)
plt.title('RUL Distribution')
plt.show()

In [ ]:
plt.figure(figsize=(12, 10))
sns.heatmap(train_df[[*active, RUL_COL]].corr(numeric_only=True), cmap='coolwarm', center=0)
plt.title('Sensors vs RUL Correlation')
plt.show()